# Notebook 04 — Preprocessing & Feature Engineering
## Machine Learning-based Late Delivery Risk Prediction in Global Supply Chain Operations
**Client:** APL Logistics (KWE Group) | **Platform:** Unified Mentor

### Objective
This notebook covers feature engineering, handling multicollinearity, train/test split, encoding, scaling, and saving all preprocessing objects. The split happens before any fitting to prevent data leakage.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

MODELS_PATH = '../models/'
os.makedirs(MODELS_PATH, exist_ok=True)

print("Imports successful.")

Imports successful.


In [2]:
data = pd.read_csv('../data/cleaned_data.csv')

df = data.copy()
print(f"Cleaned Dataset Shape: {df.shape}")
df.head()

Cleaned Dataset Shape: (180517, 28)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,Customer State,Department Name,Market,Order City,Order Country,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,1,Cardio Equipment,Brownsville,EE. UU.,Consumer,TX,Footwear,Pacific Asia,Mumbai,India,27.50,0.06,99.99,0.34,5,499.95,472.45,159.69,South Asia,Maharashtra,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,0,Shop By Sport,Littleton,EE. UU.,Consumer,CO,Golf,LATAM,San Pedro Sula,Honduras,31.99,0.16,39.99,0.29,5,199.95,167.96,48.71,Central America,Cortés,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,0,Water Sports,Littleton,EE. UU.,Consumer,CO,Fan Shop,LATAM,San Pedro Sula,Honduras,18.00,0.09,199.99,0.48,1,199.99,181.99,87.36,Central America,Cortés,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,1,Water Sports,Littleton,EE. UU.,Consumer,CO,Fan Shop,USCA,New York City,Estados Unidos,24.00,0.12,199.99,-0.24,1,199.99,175.99,-41.89,East of USA,Nueva York,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,1,Women's Apparel,Littleton,EE. UU.,Consumer,CO,Golf,USCA,New York City,Estados Unidos,10.00,0.20,50.00,0.25,1,50.00,40.00,10.00,East of USA,Nueva York,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


##  Drop Highly Correlated Columns
From the EDA correlation heatmap, we identified pairs of columns with correlation of 1.00 or 0.99 — essentially duplicate information. Keeping both causes multicollinearity, especially harmful for Logistic Regression. We keep the more interpretable column from each pair.

In [3]:
# Identified from EDA heatmap
# Benefit per order & Order Profit Per Order → correlation 1.00 → drop Order Profit Per Order
# Order Item Product Price & Product Price → correlation 1.00 → drop Product Price
# Sales per customer & Order Item Total & Sales → correlation 0.99 → keep Sales per customer

high_corr_drops = [
    'Order Profit Per Order',      # duplicate of Benefit per order
    'Product Price',               # duplicate of Order Item Product Price
    'Order Item Total',            # duplicate of Sales per customer
    'Sales'                        # duplicate of Sales per customer
]

df = df.drop(columns=high_corr_drops)
print(f"Dropped highly correlated columns: {high_corr_drops}")
print(f"Shape after dropping: {df.shape}")

Dropped highly correlated columns: ['Order Profit Per Order', 'Product Price', 'Order Item Total', 'Sales']
Shape after dropping: (180517, 24)


##  Feature Engineering
Creating new predictive indicators as specified in the project requirements. All features are derived from existing columns using pure arithmetic — no fitting required at this stage, so no leakage concern.

In [4]:
# 1. Shipping Delay Gap — how many days real shipping exceeded scheduled
df['shipping_delay_gap'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']

# 2. Shipping Pressure Index — scheduled days relative to quantity ordered
df['shipping_pressure_index'] = df['Days for shipment (scheduled)'] / (df['Order Item Quantity'] + 1)

# 3. Is Express Flag — 1 if First Class or Same Day, else 0
df['is_express'] = df['Shipping Mode'].apply(
    lambda x: 1 if x in ['First Class', 'Same Day'] else 0
)

# 4. High Discount Flag — 1 if discount rate above median
discount_median = df['Order Item Discount Rate'].median()
df['high_discount_flag'] = (df['Order Item Discount Rate'] > discount_median).astype(int)

# 5. Order Complexity Score — quantity times product price
df['order_complexity_score'] = df['Order Item Quantity'] * df['Order Item Product Price']

print("Feature Engineering Complete.")
print(f"New features added: shipping_delay_gap, shipping_pressure_index, is_express, high_discount_flag, order_complexity_score")
print(f"Shape after feature engineering: {df.shape}")
df[['shipping_delay_gap', 'shipping_pressure_index', 'is_express',
    'high_discount_flag', 'order_complexity_score']].describe()

Feature Engineering Complete.
New features added: shipping_delay_gap, shipping_pressure_index, is_express, high_discount_flag, order_complexity_score
Shape after feature engineering: (180517, 29)


,shipping_delay_gap,shipping_pressure_index,is_express,high_discount_flag,order_complexity_score
count,180517.000000,180517.000000,180517.000000,180517.000000,180517.000000
mean,0.565808,1.119354,0.208019,0.444451,203.771968
std,1.490966,0.687540,0.405892,0.496906,132.273500
min,-2.000000,0.000000,0.000000,0.000000,9.990000
25%,0.000000,0.500000,0.000000,0.000000,119.980000
50%,1.000000,1.000000,0.000000,0.000000,199.920000
75%,1.000000,2.000000,0.000000,1.000000,299.950000
max,4.000000,2.000000,1.000000,1.000000,1999.990000


##  Separate Features and Target

In [5]:
X = df.drop(columns=['Late_delivery_risk'])
y = df['Late_delivery_risk']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (180517, 28)
Target shape: (180517,)
Target distribution:
Late_delivery_risk
1    98976
0    81541
Name: count, dtype: int64


##  Train/Test Split
Splitting before any fitting. This is the most critical step to prevent data leakage. The scaler and encoders will only be fitted on X_train.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")
print(f"\ny_test distribution:\n{y_test.value_counts()}")

X_train shape: (144413, 28)
X_test shape:  (36104, 28)
y_train distribution:
Late_delivery_risk
1    79180
0    65233
Name: count, dtype: int64

y_test distribution:
Late_delivery_risk
1    19796
0    16308
Name: count, dtype: int64


##  Identify Column Types
Separating numerical and categorical columns for appropriate preprocessing.

In [8]:
# Exclude binary engineered features from scaling — they are already 0/1
binary_cols = ['is_express', 'high_discount_flag']

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols = [col for col in numerical_cols if col not in binary_cols]

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumerical columns to scale ({len(numerical_cols)}): {numerical_cols}")
print(f"\nBinary columns (no scaling needed): {binary_cols}")

Categorical columns (14): ['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Product Name', 'Shipping Mode']

Numerical columns to scale (12): ['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'shipping_delay_gap', 'shipping_pressure_index', 'order_complexity_score']

Binary columns (no scaling needed): ['is_express', 'high_discount_flag']


##  Encoding Categorical Columns
Using Label Encoding for categorical columns. Encoders are fitted on X_train only and stored in a dictionary for reuse in the Streamlit app.

In [9]:
# Encoder dictionary — fitted on X_train only
encoders = {}

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

for col in categorical_cols:
    le = LabelEncoder()
    
    # Fit on X_train only
    X_train_encoded[col] = le.fit_transform(X_train[col].astype(str))
    
    # Transform X_test — handle unseen labels safely
    X_test_encoded[col] = X_test[col].astype(str).map(
        lambda s: le.transform([s])[0] if s in le.classes_ else -1
    )
    
    encoders[col] = le

print(f"Encoding complete. Encoders fitted for {len(encoders)} columns.")
print(f"Columns encoded: {list(encoders.keys())}")

Encoding complete. Encoders fitted for 14 columns.
Columns encoded: ['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Product Name', 'Shipping Mode']


##  Scaling Numerical Columns
StandardScaler fitted on X_train only, stored in a dictionary. Applied to both X_train and X_test separately.

In [10]:
# Scaler dictionary — fitted on X_train only
scalers = {}

X_train_final = X_train_encoded.copy()
X_test_final = X_test_encoded.copy()

scaler = StandardScaler()

# Fit on X_train only
X_train_final[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])

# Transform X_test only
X_test_final[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

scalers['standard_scaler'] = scaler

print("Scaling complete.")
print(f"Columns scaled: {numerical_cols}")
print(f"\nX_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape:  {X_test_final.shape}")

Scaling complete.
Columns scaled: ['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'shipping_delay_gap', 'shipping_pressure_index', 'order_complexity_score']

X_train_final shape: (144413, 28)
X_test_final shape:  (36104, 28)


##  Save Preprocessing Objects
Saving encoders and scaler to the models/ folder. These will be loaded by the Streamlit app to apply the exact same transformations to new input data — ensuring consistent predictions without refitting.

In [11]:
# Save encoders dictionary
joblib.dump(encoders, MODELS_PATH + 'encoders.pkl')
print("Saved: models/encoders.pkl")

# Save scalers dictionary
joblib.dump(scalers, MODELS_PATH + 'scaler.pkl')
print("Saved: models/scaler.pkl")

# Save column lists for reference in app
preprocessing_config = {
    'categorical_cols': categorical_cols,
    'numerical_cols': numerical_cols,
    'binary_cols': binary_cols
}
joblib.dump(preprocessing_config, MODELS_PATH + 'preprocessing_config.pkl')
print("Saved: models/preprocessing_config.pkl")

Saved: models/encoders.pkl
Saved: models/scaler.pkl
Saved: models/preprocessing_config.pkl


##  Save Processed Data
Saving the final processed train and test sets so Notebook 05 can load them directly without repeating preprocessing.

In [12]:
X_train_final.to_csv('../data/X_train.csv', index=False)
X_test_final.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("Saved: data/X_train.csv")
print("Saved: data/X_test.csv")
print("Saved: data/y_train.csv")
print("Saved: data/y_test.csv")
print(f"\nFinal X_train shape: {X_train_final.shape}")
print(f"Final X_test shape:  {X_test_final.shape}")

Saved: data/X_train.csv
Saved: data/X_test.csv
Saved: data/y_train.csv
Saved: data/y_test.csv

Final X_train shape: (144413, 28)
Final X_test shape:  (36104, 28)


##  Preprocessing Summary

In [13]:
print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)
print(f"\nFinal dataset shape:       {df.shape}")
print(f"Training set:              {X_train_final.shape}")
print(f"Test set:                  {X_test_final.shape}")
print(f"\nFeatures engineered:       5")
print(f"Columns dropped (corr):    4")
print(f"Categorical cols encoded:  {len(categorical_cols)}")
print(f"Numerical cols scaled:     {len(numerical_cols)}")
print(f"Binary cols (unchanged):   {len(binary_cols)}")
print(f"\nSplit ratio:               80/20 stratified")
print(f"Leakage prevention:        Scaler & encoders fitted on X_train only")
print(f"\nSaved to models/:          encoders.pkl, scaler.pkl, preprocessing_config.pkl")
print(f"Saved to data/:            X_train.csv, X_test.csv, y_train.csv, y_test.csv")

PREPROCESSING SUMMARY

Final dataset shape:       (180517, 29)
Training set:              (144413, 28)
Test set:                  (36104, 28)

Features engineered:       5
Columns dropped (corr):    4
Categorical cols encoded:  14
Numerical cols scaled:     12
Binary cols (unchanged):   2

Split ratio:               80/20 stratified
Leakage prevention:        Scaler & encoders fitted on X_train only

Saved to models/:          encoders.pkl, scaler.pkl, preprocessing_config.pkl
Saved to data/:            X_train.csv, X_test.csv, y_train.csv, y_test.csv
